### Set up Imports

In [1]:
import os

import pandas as pd

from util.csv import map_ages, combine_parties_to_other, combine_vote_results
from util.web import download

### Retrieve CSV File
First, we are downloading the results which store the results by state and election type (postal or in person). This file contains the data for all the parties.

It is possible to set a path. If not, this script will call the bundeswahlleiter url to receive the election results.

Let's download the CSV file if needed and load it into a DataFrame

In [2]:
csv_file_data_by_state_and_type = "" or "data/election_results_by_state_and_type.csv"
url_by_state_and_type = "https://bundeswahlleiterin.de/dam/jcr/de996ffe-09b7-48a2-a24c-13b54a5935d4/btw21_ergebnisse_bezirksart_abs.csv"

output_file_first_votes = "data/first_votes.json"
output_file_second_votes = "data/second_votes.json"

In [3]:
if not os.path.exists(csv_file_data_by_state_and_type):
    print("Downloading file...")
    download(url_by_state_and_type, csv_file_data_by_state_and_type)
else:
    print("Using existing local file.")

Using existing local file.


Now, we can load the file into a Pandas dataframe. The first few lines contain information not needed for us.

In [4]:
# Skip first 5 rows containing comments
df = pd.read_csv(
    csv_file_data_by_state_and_type,
    skiprows=5,
    encoding='utf-8',
    sep=';'
)
df

,Nr,Land,Stimmenart,Bezirksart,Wahlberechtigte,Wähler,ungültig,gültig,CDU,SPD,...,UNABHÄNGIGE,Volt,Volksabstimmung,B*,sonstige,FAMILIE,Graue Panther,KlimalisteBW,THP,Übrige
0,99,Bundesgebiet,E,Zusammen,61172771,46707343,488483,46218860,10445920,12184096,...,13421,77594,1086,192,251,1817,961,3967,549,110874
1,99,Bundesgebiet,E,Urne,61172771,24632301,315708,24316593,5520555,6477350,...,7610,29413,324,87,99,1011,610,1354,316,48417
2,99,Bundesgebiet,E,Brief,0,22075042,172775,21902267,4925365,5706746,...,5811,48181,762,105,152,806,351,2613,233,62457
3,99,Bundesgebiet,Z,Zusammen,61172771,46707343,408956,46298387,8774920,11901558,...,22736,164300,0,0,0,0,0,0,0,0
4,99,Bundesgebiet,Z,Urne,61172771,24632301,289475,24342826,4635250,6229031,...,9509,78860,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,10,Saarland,E,Urne,755223,348874,6718,342156,91240,123571,...,0,0,0,0,0,0,0,0,0,136
98,10,Saarland,E,Brief,0,235089,4132,230957,68083,84758,...,0,0,0,0,0,0,0,0,0,163
99,10,Saarland,Z,Zusammen,755223,583963,10295,573668,135134,213777,...,0,4019,0,0,0,0,0,0,0,0
100,10,Saarland,Z,Urne,755223,348874,6519,342355,76906,123320,...,0,2308,0,0,0,0,0,0,0,0


### Remove unnecessary Data
As the CSV file contains quite a lot of data which we do not need, we should remove this.

In [5]:
df_clean = df.drop(columns=['Nr', 'Wähler', 'ungültig', 'Wahlberechtigte'])
df_clean = df_clean[
    (df_clean['Land'] != 'Bundesgebiet') &
    (df_clean['Bezirksart'] != 'Zusammen')
    ]


def map_election_type(election_type):
    return {'Urne': 'in-person', 'Brief': 'postal'}[election_type]

df_clean["Bezirksart"] = df_clean["Bezirksart"].apply(map_election_type)
mentioned_parties = ['CDU', 'SPD', 'AfD', 'FDP', 'DIE LINKE', 'GRÜNE', 'CSU']
df_clean = combine_parties_to_other(df_clean, remaining_parties=mentioned_parties)
df_clean.keys()

Index(['Land', 'Stimmenart', 'Bezirksart', 'gültig', 'CDU', 'SPD', 'AfD',
       'FDP', 'DIE LINKE', 'GRÜNE', 'CSU', 'Other'],
      dtype='object')

We should separate the data tables into two: one for second and one for first votes.  

In [6]:
df_first_votes = df_clean[df_clean['Stimmenart'] == 'E'].drop(columns=['Stimmenart'])
df_first_votes

,Land,Bezirksart,gültig,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Other
7,Schleswig-Holstein,in-person,1175398,308858,358651,87639,114095,37696,194447,0,74012
8,Schleswig-Holstein,postal,584615,157117,171650,26002,53096,17827,121186,0,37737
13,Mecklenburg-Vorpommern,in-person,595561,115845,167861,131091,41706,67291,33852,0,37915
14,Mecklenburg-Vorpommern,postal,322525,64824,93547,38886,24622,48742,28812,0,23092
19,Hamburg,in-person,488238,78816,166498,32155,41003,38009,107895,0,23862
20,Hamburg,postal,515086,100756,168333,17673,43600,34498,129433,0,20793
25,Niedersachsen,in-person,2995360,870612,1067373,228214,236191,96590,388448,0,107932
26,Niedersachsen,postal,1523212,452458,536412,63818,111611,46195,258400,0,54318
31,Bremen,in-person,175612,33531,59129,15663,12563,14466,29044,0,11216
32,Bremen,postal,151445,34461,49303,5902,9835,11457,31446,0,9041


In [7]:
df_second_votes = df_clean[df_clean['Stimmenart'] == 'Z'].drop(columns=['Stimmenart'])
df_second_votes

,Land,Bezirksart,gültig,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Other
10,Schleswig-Holstein,in-person,1176752,258708,333558,92251,146459,43620,196665,0,105491
11,Schleswig-Holstein,postal,586002,129691,160497,27315,73580,20618,126098,0,48203
16,Mecklenburg-Vorpommern,in-person,595656,102868,169454,127043,47066,60007,39145,0,50073
17,Mecklenburg-Vorpommern,postal,323203,57235,97914,38299,28489,41728,32811,0,26727
22,Hamburg,in-person,488981,68610,146958,32515,53841,36114,113694,0,37249
23,Hamburg,postal,516563,86610,151384,18022,60761,31464,136838,0,31484
28,Niedersachsen,in-person,2996638,717159,997249,262108,318912,100304,437554,0,163352
29,Niedersachsen,postal,1526583,376420,501251,74326,155726,48353,289059,0,81448
34,Bremen,in-person,176239,28186,55349,16258,16510,14176,33038,0,12722
35,Bremen,postal,151801,28313,47875,6317,13971,11176,35389,0,8760


Now, we need the second file. This file will contain the age and gender data for the vote. Same as above, one could set a path. Otherwise, it will be downloaded.

In [8]:
csv_file_data_by_gender_and_age = "" or "data/election_results_by_gender_and_age.csv"
url_by_age_and_gender = "https://bundeswahlleiterin.de/dam/jcr/a565eeb3-d324-4c0a-b921-fbfea16f8817/btw21_rws_bst2.csv"
if not os.path.exists(csv_file_data_by_gender_and_age):
    print("Downloading file...")
    download(url_by_state_and_type, url_by_age_and_gender)
else:
    print("Using existing local file.")

Using existing local file.


In [9]:
# Skip first 12 rows containing comments
df2 = pd.read_csv(
    csv_file_data_by_gender_and_age,
    skiprows=12,
    encoding='utf-8',
    sep=';'
)
df2

,Land,Erst-/Zweitstimme,Geschlecht,Geburtsjahresgruppe,Summe,Ungültig,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Sonstige,dar. FREIE WÄHLER,dar. dieBasis
0,Bund,1,Summe,Summe,46854508,492495,10451524.0,12234690,4695611,4042951,2307536,6469081.0,2788048.0,3372572,1316686,734011.0
1,Bund,1,Summe,1997 – 2003,3524799,25512,434605.0,667343,218003,592862,276324,816458.0,132261.0,361432,106749,48258.0
2,Bund,1,Summe,1987 – 1996,5992701,44955,835239.0,1203503,588419,718457,407852,1289831.0,253494.0,650951,224446,111570.0
3,Bund,1,Summe,1977 – 1986,6466965,51452,1197212.0,1342590,880187,609213,323578,1081171.0,332210.0,649352,242994,155238.0
4,Bund,1,Summe,1962 – 1976,12604097,111747,2721637.0,3171648,1559505,1051192,523633,1738183.0,727673.0,998879,416490,257493.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
793,TH,2,w,1987 – 1996,61050,841,5862.0,8564,13845,6674,5988,9233.0,NaN,10043,1780,1560.0
794,TH,2,w,1977 – 1986,92647,1219,12591.0,15827,24309,10909,7380,8027.0,NaN,12386,2694,2991.0
795,TH,2,w,1962 – 1976,165023,1721,27143.0,35507,38612,16914,17773,9896.0,NaN,17458,4804,3914.0
796,TH,2,w,1952 – 1961,138490,1434,27692.0,40957,24443,9838,18333,5627.0,NaN,10166,2986,2534.0


In [10]:
df2_clean = df2.drop(columns=['Ungültig', 'dar. FREIE WÄHLER', 'dar. dieBasis'])
df2_clean = df2_clean[
    (df2_clean['Land'] != 'Bund')
    & (df2_clean['Geschlecht'] != 'Summe')
    & (df2_clean['Geburtsjahresgruppe'] != 'Summe')
    ]
df2_clean['Geburtsjahresgruppe'] = df2_clean['Geburtsjahresgruppe'].apply(map_ages)
df2_clean = df2_clean.rename(columns={'Erst-/Zweitstimme': 'Stimmenart'})
df2_clean

,Land,Stimmenart,Geschlecht,Geburtsjahresgruppe,Summe,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Sonstige
50,SH,1,m,18-24,66529,9478.0,12901,3319,14435,3857,16164.0,NaN,5681
51,SH,1,m,25-34,96579,14851.0,20904,10100,15494,5186,20209.0,NaN,9256
52,SH,1,m,35-44,107371,23770.0,27281,13607,10462,4101,17679.0,NaN,9922
53,SH,1,m,45-54,245329,66001.0,68443,24942,22852,6835,37377.0,NaN,17108
54,SH,1,m,55-64,150261,38538.0,52157,11096,10622,3944,24632.0,NaN,7639
...,...,...,...,...,...,...,...,...,...,...,...,...,...
793,TH,2,w,25-34,61050,5862.0,8564,13845,6674,5988,9233.0,NaN,10043
794,TH,2,w,35-44,92647,12591.0,15827,24309,10909,7380,8027.0,NaN,12386
795,TH,2,w,45-54,165023,27143.0,35507,38612,16914,17773,9896.0,NaN,17458
796,TH,2,w,55-64,138490,27692.0,40957,24443,9838,18333,5627.0,NaN,10166


Separate dataframe into first and second votes again.

In [11]:
df2_first_votes = df2_clean[df2_clean['Stimmenart'] == 1].drop(columns=['Stimmenart'])
df2_first_votes

,Land,Geschlecht,Geburtsjahresgruppe,Summe,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Sonstige
50,SH,m,18-24,66529,9478.0,12901,3319,14435,3857,16164.0,NaN,5681
51,SH,m,25-34,96579,14851.0,20904,10100,15494,5186,20209.0,NaN,9256
52,SH,m,35-44,107371,23770.0,27281,13607,10462,4101,17679.0,NaN,9922
53,SH,m,45-54,245329,66001.0,68443,24942,22852,6835,37377.0,NaN,17108
54,SH,m,55-64,150261,38538.0,52157,11096,10622,3944,24632.0,NaN,7639
...,...,...,...,...,...,...,...,...,...,...,...,...
772,TH,w,25-34,61050,8288.0,9875,14497,5537,7933,7394.0,NaN,6513
773,TH,w,35-44,92647,16567.0,16530,23837,8807,9192,6330.0,NaN,9963
774,TH,w,45-54,165023,33638.0,35604,38528,13662,18890,8909.0,NaN,13964
775,TH,w,55-64,138490,32402.0,39304,24421,7467,19298,4951.0,NaN,8993


In [12]:
df2_second_votes = df2_clean[df2_clean['Stimmenart'] == 2].drop(columns=['Stimmenart'])
df2_second_votes

,Land,Geschlecht,Geburtsjahresgruppe,Summe,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Sonstige
71,SH,m,18-24,66529,5550.0,11466,3653,17629,4253,14872.0,NaN,8512
72,SH,m,25-34,96579,11268.0,17128,10723,18734,5871,19996.0,NaN,12469
73,SH,m,35-44,107371,17920.0,23417,13783,14787,4559,20032.0,NaN,12458
74,SH,m,45-54,245329,53882.0,64224,25932,31064,7755,40332.0,NaN,20986
75,SH,m,55-64,150261,32503.0,49395,11444,15210,5993,25324.0,NaN,9086
...,...,...,...,...,...,...,...,...,...,...,...,...
793,TH,w,25-34,61050,5862.0,8564,13845,6674,5988,9233.0,NaN,10043
794,TH,w,35-44,92647,12591.0,15827,24309,10909,7380,8027.0,NaN,12386
795,TH,w,45-54,165023,27143.0,35507,38612,16914,17773,9896.0,NaN,17458
796,TH,w,55-64,138490,27692.0,40957,24443,9838,18333,5627.0,NaN,10166


In [13]:
first_vote_results = combine_vote_results(df_first_votes, df2_first_votes, vote_type="1")
len(first_vote_results)

by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-W' not found in state mapping. Skipping it
by_demographic['Land']='BE-W' not found in state mapping. Skipping it
by_demographic['Land

3072

In [14]:
second_vote_results = combine_vote_results(df_second_votes, df2_second_votes, vote_type="2")
len(second_vote_results)

by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-O' not found in state mapping. Skipping it
by_demographic['Land']='BE-W' not found in state mapping. Skipping it
by_demographic['Land']='BE-W' not found in state mapping. Skipping it
by_demographic['Land

3072

Save results in json files

In [15]:
dcc = {}
total_votes = 0
for entry in second_vote_results:
    if entry.party in dcc:
        dcc[entry.party] += entry.votes
    else:
        dcc[entry.party] = 0
    total_votes += entry.votes
for d in dcc:
    dcc[d] /= total_votes
dcc

{'CDU': np.float64(0.1894498659393563),
 'SPD': np.float64(0.2568948414869912),
 'AfD': np.float64(0.10381386676590959),
 'FDP': np.float64(0.11402727703715887),
 'DIE LINKE': np.float64(0.048662086201665795),
 'GRÜNE': np.float64(0.14698884081613364),
 'CSU': np.float64(0.051898719495346635),
 'Sonstige': np.float64(0.08731884365289153)}

In [16]:
from dataclasses import asdict
import json

with open(output_file_first_votes, 'w') as f:
    json.dump([asdict(v) for v in first_vote_results], f, indent=4)
with open(output_file_second_votes, 'w') as f:
    json.dump([asdict(v) for v in second_vote_results], f, indent=4)

Compare the two different results

In [17]:
from dataclasses import asdict

approximated_result_data = [asdict(entry) for entry in second_vote_results]
approximated_result_data = pd.DataFrame(approximated_result_data)

approx_df = approximated_result_data.pivot_table(index='state', columns='party', values='votes', aggfunc='sum',
                                                 fill_value=0)

# Reset the index to convert 'state' back into a column, and rename it to 'Land'
approx_df = approx_df.reset_index().rename(columns={'state': 'Land'})
exact_df = (
    df_second_votes
    .groupby("Land", as_index=False)[df_second_votes.columns.difference(["Land", "Bezirksart", "gültig"])]
    .sum()
)

In [18]:
exact_df_indexed = exact_df.set_index("Land")
exact_df_indexed = exact_df_indexed.rename(columns={'Other': 'Sonstige'})
approx_df_indexed = approx_df.set_index("Land")

In [20]:
(approx_df_indexed - exact_df_indexed).abs()

,AfD,CDU,CSU,DIE LINKE,FDP,GRÜNE,SPD,Sonstige
Land,,,,,,,,
Baden-Württemberg,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Bayern,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Berlin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Brandenburg,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Bremen,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hamburg,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hessen,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Mecklenburg-Vorpommern,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Niedersachsen,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
